# 10 — AHR Go engine interface

Connect a frozen AHR player to a Go GUI or match controller through standard input and output. Run All checks the protocol, legality, undo, clock budgeting and a real subprocess conversation; it does not train a model or open a network service.

**Inputs:** tagged definitions from notebooks 01 and 06; an injected callable `player(state, time_budget_s)` returning a dictionary with an integer `action`. The default checks use a clearly labelled deterministic legal fixture. A trained player is supplied explicitly through the factory interface below.

**Outputs:** this executed notebook, `reports/10_gtp_validation.json`, and an HTML preview. Expected default runtime: under one minute. There is no training checkpoint in this notebook. For an interrupted GUI session, restart the same frozen player and replay `boardsize`, `clear_board`, `komi`, and all successful `play` commands, then restore `time_settings`/`time_left`; this reconstructs full superko history. Undo stores every prior board, history and capture count in memory. It does not rewind model randomness, komi settings or clocks.

**Rules:** area scoring; positional superko; suicide forbidden; pass exempt from superko; two alternating passes end play. Dead stones must be captured before passing. `final_score` is available only after termination; it never adjudicates dead stones. Sizes 2–9 are supported by the interface; an injected model may narrow that support. Handicap and SGF commands are not advertised.

The [official GTP v2 draft specification](https://www.lysator.liu.se/~gunnar/gtp/gtp2-spec-draft2/gtp2-spec.html) governs framing, coordinates, command discovery and clocks. This implementation accepts either requested colour, as GTP requires. It changes the player to move without inserting a pass; a repeated pass by the same colour starts a new pass streak. On an already terminal board `genmove` acknowledges `pass` without adding a move, and `play` fails. Time limits are cooperative; an external match controller measures and enforces wall time.

Version 2 adds a pinned GNU Go evaluation adapter with full-history replay and bounded subprocess cleanup. The AHR learning and move selection algorithms remain in notebooks 08–09. Protocol checks below take under two minutes; the reference and latency companion is `reports/10_v2_reference_and_latency.ipynb`.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, math, time, copy, io, re, subprocess
import nbformat

PROJECT = Path(os.environ.get("AHR_GO_PROJECT", Path.cwd())).resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
NOTEBOOK_PATH = PROJECT / "notebooks/10_gtp_engine_interface.ipynb"
SOURCE_HASHES = {}
for filename in ("01_go_rules_and_independent_verifier.ipynb", "06_ahr_go_search_engine.ipynb"):
    upstream = nbformat.read(PROJECT / "notebooks" / filename, as_version=4)
    definitions = [cell.source for cell in upstream.cells
                   if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])]
    if not definitions:
        raise RuntimeError(f"Missing tagged definitions: {filename}")
    for index, source in enumerate(definitions):
        exec(compile(source, f"{filename}:ahr-export-{index}", "exec"), globals())
    SOURCE_HASHES[filename] = hashlib.sha256("\n".join(definitions).encode()).hexdigest()
print("Loaded definitions only; upstream training and validation cells were not executed.")

## Parse and frame the protocol

Command names are case sensitive; colours and vertices are case insensitive. Coordinates run from A1 at the lower left to J9 at the upper right, omitting I. Comments, tabs, carriage returns and discarded ASCII control characters follow the protocol preprocessing rules. Invalid commands return a framed error and leave engine state unchanged.

In [ ]:
from dataclasses import replace
from contextlib import redirect_stdout
from collections.abc import Mapping
import io, sys, json, math, time, re, copy, hashlib
from pathlib import Path

GTP_ENGINE_VERSION = "ahr-gtp-1.0"
GTP_COLUMNS = "ABCDEFGHJKLMNOPQRSTUVWXYZ"
GTP_COMMANDS = (
    "protocol_version", "name", "version", "known_command", "list_commands", "quit",
    "boardsize", "clear_board", "komi", "play", "genmove", "undo", "showboard",
    "final_score", "time_settings", "time_left", "ahr-rules", "ahr-last-search",
)


class GTPCommandError(ValueError):
    pass


def gtp_uint(token):
    if not isinstance(token, str) or re.fullmatch(r"[0-9]+", token) is None:
        raise GTPCommandError("syntax error: expected unsigned integer")
    # Bound string length before conversion, including strings of leading zeroes.
    normalized = token.lstrip("0") or "0"
    if len(normalized) > 10 or int(normalized) > 2**31 - 1:
        raise GTPCommandError("syntax error: integer outside GTP range")
    return int(normalized)


def gtp_color(token):
    color = {"b": BLACK, "black": BLACK, "w": WHITE, "white": WHITE}.get(token.lower())
    if color is None:
        raise GTPCommandError("syntax error: invalid color")
    return color


def gtp_vertex_to_action(vertex, size):
    if vertex.lower() == "pass":
        return size * size
    match = re.fullmatch(r"([A-Za-z])([0-9]+)", vertex)
    if match is None or match.group(1).upper() not in GTP_COLUMNS[:size]:
        raise GTPCommandError("syntax error: invalid vertex")
    row_number = gtp_uint(match.group(2))
    if not 1 <= row_number <= size:
        raise GTPCommandError("syntax error: invalid vertex")
    column = GTP_COLUMNS.index(match.group(1).upper())
    return (size - row_number) * size + column


def gtp_action_to_vertex(action, size):
    if type(action) is not int or not 0 <= action <= size * size:
        raise GTPCommandError("invalid generated action")
    if action == size * size:
        return "pass"
    row, column = divmod(action, size)
    return f"{GTP_COLUMNS[column]}{size-row}"


def gtp_parse_line(raw):
    if not isinstance(raw, str):
        raise GTPCommandError("syntax error: input must be text")
    clean = "".join(c for c in raw if c in "\t\n" or (ord(c) >= 32 and ord(c) != 127))
    if "\n" in clean.rstrip("\n"):
        raise GTPCommandError("syntax error: expected one command line")
    clean = clean.split("#", 1)[0].replace("\t", " ").strip(" \n")
    if not clean:
        return None
    if any(ord(c) > 126 for c in clean):
        raise GTPCommandError("syntax error: standard commands require ASCII")
    parts = clean.split()
    command_id = None
    if re.fullmatch(r"[0-9]+", parts[0]):
        command_id = str(gtp_uint(parts.pop(0)))
    if not parts:
        return command_id, "", []
    return command_id, parts[0], parts[1:]


def gtp_response(success, command_id=None, payload=""):
    marker = "=" if success else "?"
    payload = str(payload).replace("\r", "").replace("\t", " ").strip("\n")
    # Payloads cannot inject a premature frame terminator or ASCII controls.
    payload = "\n".join(line for line in payload.split("\n") if line.strip())
    payload = "".join(c for c in payload if c == "\n" or 32 <= ord(c) < 127)
    return marker + (command_id or "") + (" " + payload if payload else "") + "\n\n"

## Maintain board state and thinking time

Generated moves pass both the authoritative rules implementation and notebook 06's independent transition checker before being committed. The engine stores diagnostics separately from the GTP result: a legal move carries no proof of winning. A player error or malformed action never causes an implicit move.

`time_settings` and `time_left` guide a conservative budget capped by `default_time_budget_s`. Canadian overtime and absolute time are supported. The controller remains responsible for timeout adjudication; an overrun is reported by `ahr-last-search`, not converted into a proved Go loss.

In [ ]:
class AHRGTPEngine:
    """Synchronous GTP interface for a frozen player; no training or network service."""
    def __init__(self, player, board_size=9, komi=7.5, default_time_budget_s=1.0,
                 supported_sizes=tuple(range(2, 10)), player_identity="injected_frozen_player"):
        if not callable(player):
            raise TypeError("player must be callable")
        if not supported_sizes or any(type(n) is not int or not 2 <= n <= 9 for n in supported_sizes):
            raise ValueError("Supported sizes must be integers from 2 to 9")
        if type(board_size) is not int or board_size not in supported_sizes:
            raise ValueError("Unsupported board size")
        if type(default_time_budget_s) not in (int, float) or not math.isfinite(default_time_budget_s) or default_time_budget_s < 0:
            raise ValueError("Default time budget must be finite and nonnegative")
        self.player = player
        self.supported_sizes = tuple(sorted(set(supported_sizes)))
        self.default_time_budget_s = float(default_time_budget_s)
        self.player_identity = str(player_identity)
        self.state = initial_state(board_size, komi)
        self.undo_stack = []
        self.captures = {BLACK: 0, WHITE: 0}
        self.time_control = None
        self.clocks = {BLACK: None, WHITE: None}
        self.last_generation = None
        self.closed = False

    def session_snapshot(self):
        """Protocol state only; injected player weights/RNG are not serialized."""
        return copy.deepcopy({"state": self.state, "undo_stack": self.undo_stack,
                              "captures": self.captures, "time_control": self.time_control,
                              "clocks": self.clocks, "last_generation": self.last_generation,
                              "closed": self.closed})

    def _restore_snapshot(self, snapshot):
        for key, value in snapshot.items():
            setattr(self, key, value)

    def _requested_state(self, color):
        if color == self.state.to_play:
            return self.state
        # GTP accepts consecutive same-color moves, without inserting a pass.
        # A second pass by the same color is not an alternating two-pass ending.
        return replace(self.state, to_play=color, consecutive_passes=0)

    def _commit_move(self, parent, action, child):
        if not verify_go_emission(parent, action, child):
            raise GTPCommandError("independent move verification failed")
        captured = parent.board.count(-parent.to_play) - child.board.count(-parent.to_play)
        self.undo_stack.append((self.state, copy.deepcopy(self.captures)))
        self.captures[parent.to_play] += captured
        self.state = child

    def _reset_board(self, size):
        self.state = initial_state(size, self.state.komi)
        self.undo_stack = []
        self.captures = {BLACK: 0, WHITE: 0}
        self.last_generation = None

    def time_budget(self, color):
        clock = self.clocks[color]
        if self.time_control is None or self.time_control[1] > 0 and self.time_control[2] == 0:
            return self.default_time_budget_s
        if clock is None:
            return self.default_time_budget_s
        remaining, stones = clock
        if stones == 0 and remaining <= 0 and self.time_control[1] > 0:
            remaining, stones = float(self.time_control[1]), self.time_control[2]
        moves = stones if stones else max(10, (self.state.size**2 - self.state.move_number) // 2)
        return min(self.default_time_budget_s, max(0.0, remaining * 0.9 / max(1, moves)))

    def _charge_clock(self, color, elapsed):
        if self.time_control is None or self.clocks[color] is None:
            return
        _, overtime, overtime_stones = self.time_control
        if overtime > 0 and overtime_stones == 0:
            return
        remaining, stones = self.clocks[color]
        remaining -= elapsed
        if stones:
            stones -= 1
            if stones == 0 and remaining >= 0:
                remaining, stones = float(overtime), overtime_stones
        elif remaining <= 0 and overtime > 0:
            remaining += overtime
            stones = max(0, overtime_stones - 1)
            if stones == 0 and remaining >= 0:
                remaining, stones = float(overtime), overtime_stones
        self.clocks[color] = (max(0.0, remaining), stones)

    def _generate(self, color):
        if self.state.terminal:
            self.last_generation = {"status": "TERMINAL", "action": None,
                "strategic_status": "UNKNOWN", "reason": "terminal_pass_acknowledgment_without_new_move"}
            return "pass"
        parent = self._requested_state(color)
        budget = self.time_budget(color)
        started = time.monotonic()
        if budget == 0:
            result = {"action": parent.pass_action, "status": "LEGAL",
                      "strategic_status": "UNKNOWN", "reason": "zero_budget_legal_pass"}
        else:
            # Keep accidental player prints off the protocol output stream.
            with redirect_stdout(sys.stderr):
                result = self.player(parent, time_budget_s=budget)
        if not isinstance(result, Mapping) or type(result.get("action")) is not int:
            raise GTPCommandError("invalid generated action")
        action = result["action"]
        try:
            child = play(parent, action)
        except IllegalMove as error:
            raise GTPCommandError("illegal generated move") from error
        self._commit_move(parent, action, child)
        elapsed = time.monotonic() - started
        self._charge_clock(color, elapsed)
        self.last_generation = {"action": action, "vertex": gtp_action_to_vertex(action, parent.size),
            "legality_verified": True, "strategic_status": "UNKNOWN", "elapsed_s": elapsed,
            "time_budget_s": budget, "deadline_exceeded": elapsed > budget,
            "player_reported_status": str(result.get("status", "UNKNOWN")),
            "player_identity": self.player_identity}
        checkpoint_metadata = getattr(self.player, "ahr_checkpoint_metadata", None)
        if checkpoint_metadata is not None:
            self.last_generation["checkpoint"] = copy.deepcopy(checkpoint_metadata)
        return self.last_generation["vertex"]

    def _dispatch(self, command, args):
        arity = {"known_command": 1, "boardsize": 1, "komi": 1, "play": 2,
                 "genmove": 1, "time_settings": 3, "time_left": 3}
        if command not in GTP_COMMANDS:
            raise GTPCommandError("unknown command")
        if len(args) != arity.get(command, 0):
            raise GTPCommandError("syntax error: wrong number of arguments")
        if command == "protocol_version": return "2"
        if command == "name": return "AHR Go"
        if command == "version": return GTP_ENGINE_VERSION
        if command == "known_command": return "true" if args[0] in GTP_COMMANDS else "false"
        if command == "list_commands": return "\n".join(GTP_COMMANDS)
        if command == "quit":
            self.closed = True
            return ""
        if command == "boardsize":
            size = gtp_uint(args[0])
            if size not in self.supported_sizes:
                raise GTPCommandError("unacceptable size")
            self._reset_board(size)
        elif command == "clear_board":
            self._reset_board(self.state.size)
        elif command == "komi":
            try:
                komi = float(args[0])
            except ValueError as error:
                raise GTPCommandError("syntax error: invalid komi") from error
            if not math.isfinite(komi) or abs(komi) > 3.4028234663852886e38:
                raise GTPCommandError("syntax error: komi must be a finite GTP float")
            self.state = replace(self.state, komi=komi)
        elif command == "play":
            color = gtp_color(args[0])
            action = gtp_vertex_to_action(args[1], self.state.size)
            if self.state.terminal:
                raise GTPCommandError("illegal move")
            parent = self._requested_state(color)
            try:
                child = play(parent, action)
            except IllegalMove as error:
                raise GTPCommandError("illegal move") from error
            self._commit_move(parent, action, child)
        elif command == "genmove":
            return self._generate(gtp_color(args[0]))
        elif command == "undo":
            if not self.undo_stack:
                raise GTPCommandError("cannot undo")
            prior, captures = self.undo_stack.pop()
            self.state = replace(prior, komi=self.state.komi)
            self.captures = captures
            self.last_generation = None
        elif command == "showboard":
            letters = " ".join(GTP_COLUMNS[:self.state.size])
            rows = ["   " + letters]
            for row in range(self.state.size):
                stones = " ".join({BLACK: "X", WHITE: "O", EMPTY: "."}[stone]
                    for stone in self.state.board[row*self.state.size:(row+1)*self.state.size])
                rows.append(f"{self.state.size-row:2d} {stones} {self.state.size-row}")
            rows.append("   " + letters)
            return "\n".join(rows)
        elif command == "final_score":
            if not self.state.terminal:
                raise GTPCommandError("cannot score")
            score = area_score(self.state)
            return "0" if score == 0 else f"{'B' if score > 0 else 'W'}+{abs(score):g}"
        elif command == "time_settings":
            main, overtime, stones = map(gtp_uint, args)
            self.time_control = (main, overtime, stones)
            initial = (float(overtime), stones) if main == 0 and overtime > 0 and stones else (float(main), 0)
            self.clocks = {BLACK: initial, WHITE: initial}
        elif command == "time_left":
            color = gtp_color(args[0])
            self.clocks[color] = (float(gtp_uint(args[1])), gtp_uint(args[2]))
        elif command == "ahr-rules":
            return json.dumps(RULESET, sort_keys=True, separators=(",", ":"))
        elif command == "ahr-last-search":
            return json.dumps(self.last_generation, sort_keys=True, separators=(",", ":"))
        return ""

    def handle_line(self, raw):
        command_id = None
        if self.closed:
            return None
        snapshot = self.session_snapshot()
        try:
            parsed = gtp_parse_line(raw)
            if parsed is None:
                return None
            command_id, command, args = parsed
            payload = self._dispatch(command, args)
            return gtp_response(True, command_id, payload)
        except (GTPCommandError, IllegalMove) as error:
            self._restore_snapshot(snapshot)
            return gtp_response(False, command_id, str(error))
        except Exception:
            self._restore_snapshot(snapshot)
            return gtp_response(False, command_id, "engine error; state unchanged")

    def run(self, input_stream=None, output_stream=None):
        source = sys.stdin if input_stream is None else input_stream
        destination = sys.stdout if output_stream is None else output_stream
        for line in source:
            response = self.handle_line(line)
            if response is not None:
                destination.write(response)
                destination.flush()
            if self.closed:
                break

## Launch from notebooks only

`make_gtp_subprocess_command` returns an argument list for `subprocess.Popen` or a match runner. Its `python -c` bootstrap is defined here; no helper source file is created. A factory consists of explicitly tagged notebook definitions, a function name and JSON arguments. The function must return a frozen player callable. Loaded definitions and factory initialization write diagnostics to stderr.

The default factory is a deterministic **protocol fixture**, not a trained AHR. For the existing constraint model, `make_csp_gtp_player(project, run_id, device)` loads notebook 05's verified checkpoint and uses notebook 06 search. For the spatial Go model, `make_go_gtp_subprocess_command(PROJECT, run_id, device="cuda:0")` loads notebook 08 with the exact chosen checkpoint and permits only 9×9. Missing, invalid or incompatible checkpoints fail explicitly. This notebook never chooses a checkpoint silently.

The Go factory imports into a private namespace and rehydrates the complete incoming state there. This preserves type compatibility and superko history when the engine and model were loaded by different notebook kernels or import helpers.

In [ ]:
def gtp_fixture_player(state, time_budget_s):
    """Deterministic legal fixture for protocol checks, never a strength baseline."""
    return {"action": legal_actions(state)[0], "status": "FIXTURE", "strategic_status": "UNKNOWN"}


def make_csp_gtp_player(project, run_id, device="cpu", search_config=None):
    model, codec, metadata = load_pretrained_ahr(Path(project), run_id, device=device, selection="best")
    def player(state, time_budget_s):
        return choose_ahr_go_move(state, model, codec, time_budget_s=time_budget_s,
                                  search_config=search_config)
    return player


def make_gtp_subprocess_command(project, *, factory_notebooks=(), factory_name=None,
                                factory_kwargs=None, engine_kwargs=None):
    """Explicit trusted notebook factory; returns argv, not a shell command."""
    project = Path(project).resolve()
    notebooks = []
    for filename in factory_notebooks:
        path = (project / "notebooks" / filename).resolve()
        if path.parent != project / "notebooks" or path.suffix != ".ipynb" or not path.is_file():
            raise ValueError("Factory notebooks must be existing direct children of project/notebooks")
        notebooks.append(path.name)
    if factory_name is not None and (not isinstance(factory_name, str) or not factory_name.isidentifier()):
        raise ValueError("Factory name must be a Python identifier")
    specification = {"project": str(project), "factory_notebooks": notebooks,
        "factory_name": factory_name, "factory_kwargs": factory_kwargs or {},
        "engine_kwargs": engine_kwargs or {}}
    # This executable bootstrap remains authored in the notebook itself.
    bootstrap = r"""
import os, sys, json, hashlib, math, time, copy, io
from pathlib import Path
from contextlib import redirect_stdout
import nbformat
spec = json.loads(sys.argv[1])
PROJECT = Path(spec["project"])
sequence = ["01_go_rules_and_independent_verifier.ipynb", "06_ahr_go_search_engine.ipynb"]
sequence += spec["factory_notebooks"]
sequence += ["10_gtp_engine_interface.ipynb"]
with redirect_stdout(sys.stderr):
    for filename in dict.fromkeys(sequence):
        notebook = nbformat.read(PROJECT / "notebooks" / filename, as_version=4)
        definitions = [c.source for c in notebook.cells if c.cell_type == "code"
                       and "ahr-export" in c.metadata.get("tags", [])]
        if not definitions:
            raise RuntimeError("No tagged definitions in " + filename)
        for index, source in enumerate(definitions):
            exec(compile(source, filename + ":export:" + str(index), "exec"), globals())
    player = (globals()[spec["factory_name"]](**spec["factory_kwargs"])
              if spec["factory_name"] else gtp_fixture_player)
    engine = AHRGTPEngine(player, **spec["engine_kwargs"])
engine.run()
"""
    return [sys.executable, "-u", "-c", bootstrap, json.dumps(specification, allow_nan=False)]


def _gtp_bind_go_player(namespace, model, search_config=None):
    configuration = namespace["GoAHRSearchConfig"](**search_config) if search_config is not None else None
    def player(state, time_budget_s):
        if state.size != 9:
            raise GTPCommandError("Go-trained player requires a 9x9 board")
        # Notebook imports intentionally create distinct Python class identities.
        # Rehydrate the complete immutable state inside the model's namespace.
        private_state = namespace["GoState"].from_dict(state.to_dict())
        return namespace["choose_go_ahr_move"](private_state, model,
            time_budget_s=time_budget_s, search_config=configuration)
    return player


def make_go_gtp_player(project, run_id, device="cpu", selection="best", search_config=None):
    """Load the exact Go checkpoint in an isolated definition-only namespace."""
    import nbformat
    project = Path(project).resolve()
    namespace = {"__name__": "__ahr_gtp_go_player__", "nbformat": nbformat,
                 "__file__": str(project / "notebooks/08_go_specific_adaptation.ipynb")}
    with redirect_stdout(sys.stderr):
        for number in (*range(7), 8):
            matches = list((project / "notebooks").glob(f"{number:02d}_*.ipynb"))
            if len(matches) != 1:
                raise ValueError(f"Expected exactly one upstream notebook {number:02d}")
            notebook = nbformat.read(matches[0], as_version=4)
            definitions = [cell.source for cell in notebook.cells if cell.cell_type == "code"
                           and "ahr-export" in cell.metadata.get("tags", [])]
            if not definitions:
                raise ValueError("No tagged definitions in " + matches[0].name)
            for index, source in enumerate(definitions):
                exec(compile(source, matches[0].name + f":export:{index}", "exec"), namespace)
        model, metadata = namespace["load_go_ahr"](project, run_id, device=device, selection=selection)
    if model.config.board_size != 9:
        raise ValueError("Selected Go checkpoint must have a 9x9 model configuration")
    player = _gtp_bind_go_player(namespace, model, search_config)
    player.ahr_checkpoint_metadata = {key: metadata[key] for key in
        ("run_id", "selection", "checkpoint_file", "checkpoint_sha256", "model_sha256", "optimizer_step")}
    return player


def make_go_gtp_subprocess_command(project, run_id, device="cpu", selection="best",
                                   default_time_budget_s=0.25, search_config=None):
    project = Path(project).resolve()
    definitions = []
    for number in (*range(7), 8):
        matches = list((project / "notebooks").glob(f"{number:02d}_*.ipynb"))
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one upstream notebook {number:02d}")
        definitions.append(matches[0].name)
    return make_gtp_subprocess_command(project,
        factory_name="make_go_gtp_player",
        factory_kwargs={"project": str(project), "run_id": run_id, "device": device,
                        "selection": selection, "search_config": search_config},
        engine_kwargs={"board_size": 9, "supported_sizes": [9],
                       "default_time_budget_s": default_time_budget_s,
                       "player_identity": f"go_ahr:{run_id}:{selection}"})


## Frozen reference engine for evaluation

GNU Go 3.8 is installed under `.venv/third_party/gnugo` from Debian package `gnugo_3.8-13_amd64.deb`. Its executable and package hashes are pinned below. To reinstall on compatible Debian amd64: download that exact package with `apt-get download gnugo=3.8-13`, verify the package hash, and unpack it with `dpkg-deb -x` into that directory. No system service or training teacher is installed.

The [GNU Go rule options](https://www.gnu.org/s/gnugo/gnugo_3.html) provide Chinese area play, positional superko, forbidden suicide and capture-all-dead. The [GTP manual](https://www.gnu.org/s/gnugo/gnugo_19.html) defines the command protocol. We disable resignation, replay the complete chronological game, independently validate every returned move, and use **our final area scorer**, never GNU Go's dead-stone adjudication. Capture-all-dead is an engine behavior setting, not proof that every dead group is captured before passing. Both players are judged on the same actual final board after two passes.

`arena_gnugo_player` is a frozen, evaluation-only opponent. Its preparation hook requires saved opening and game actions; unordered board-history sets are deliberately insufficient. Each move starts a fresh process with a state-derived fixed seed, so exact recovery needs no hidden engine state. Replay, startup and cleanup count against the wall-clock move budget. Timeout kills/reaps the process and leaves the game UNKNOWN; it does not become a loss. External engines have no meaningful AHR node/model-call budget, so these matches use an empty work-budget contract and matched wall-clock limits. GNU Go's level is fixed, not an advertised Elo or dan rank.


In [ ]:
# GNU Go is an evaluation opponent only. It never supplies training labels or AHR moves.
import subprocess, hashlib, time, json, math
GNUGO10_BINARY_RELATIVE = ".venv/third_party/gnugo/usr/bin/gnugo"
GNUGO10_BINARY_SHA256 = "b530fbddbfd613345ea31c45ddf81be22be4328782a8a7b09f5d2d80a6613e93"
GNUGO10_PACKAGE_SHA256 = "f0feff31a6d743e6c6863c4cbabde26ee88c31918c230b57b83b6be1db78c3e5"
GNUGO10_RULE_FLAGS = ["--chinese-rules", "--positional-superko", "--forbid-suicide",
                     "--capture-all-dead", "--never-resign"]

def gnugo10_binary_identity(project):
    path = Path(project).resolve() / GNUGO10_BINARY_RELATIVE
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != GNUGO10_BINARY_SHA256:
        raise ValueError("Missing or changed pinned GNU Go binary; restore the documented package before evaluation")
    version = subprocess.run([str(path), "--version"], text=True, capture_output=True,
                             timeout=3, check=True).stdout.splitlines()[0]
    if version != "GNU Go 3.8": raise ValueError("GNU Go version mismatch")
    return {"binary_relative": GNUGO10_BINARY_RELATIVE, "binary_sha256": GNUGO10_BINARY_SHA256,
            "version": version, "debian_package": "gnugo_3.8-13_amd64.deb",
            "package_sha256": GNUGO10_PACKAGE_SHA256}

def gnugo10_exchange(project, commands, *, level=1, seed=20260924, timeout_s=2.0):
    """Bounded one-process transaction; timeout kills and waits for the child before raising."""
    if type(level) is not int or not 0 <= level <= 10: raise ValueError("GNU Go level must be 0..10")
    if type(seed) is not int or not 0 <= seed <= 2**31-1: raise ValueError("Invalid deterministic seed")
    if not math.isfinite(timeout_s) or timeout_s <= 0: raise ValueError("Invalid timeout")
    if len(commands) > 4096 or any(not isinstance(c,str) or len(c)>128 or any(x in c for x in "\r\n") for c in commands):
        raise ValueError("Bounded single-line GTP commands required")
    path = Path(project).resolve()/GNUGO10_BINARY_RELATIVE
    # The factory checks the immutable executable once; the exchange checks again to refuse replacement.
    if hashlib.sha256(path.read_bytes()).hexdigest() != GNUGO10_BINARY_SHA256:
        raise ValueError("GNU Go executable changed")
    argv=[str(path),"--mode","gtp","--quiet","--level",str(level),"--min-level",str(level),
          "--max-level",str(level),"--seed",str(seed),*GNUGO10_RULE_FLAGS]
    request="".join(f"{i+1} {command}\n" for i,command in enumerate([*commands,"quit"]))
    start=time.perf_counter()
    try:
        process=subprocess.run(argv,input=request,text=True,capture_output=True,timeout=timeout_s,
                               cwd=Path(project),check=True)
    except subprocess.TimeoutExpired as error:
        raise TimeoutError("GNU Go command budget exceeded; child killed and reaped; result UNKNOWN") from error
    elapsed=time.perf_counter()-start
    frames=process.stdout.replace("\r","").strip().split("\n\n")
    if len(frames)!=len(commands)+1: raise RuntimeError("Malformed GNU Go response framing")
    payload=[]
    for index,frame in enumerate(frames,1):
        match=re.fullmatch(r"([=?])(\d+)(?:[ ](.*))?",frame,re.DOTALL)
        if match is None or int(match.group(2))!=index: raise RuntimeError("GNU Go response ID mismatch")
        if match.group(1)!="=": raise GTPCommandError("GNU Go rejected command: "+(match.group(3) or ""))
        payload.append((match.group(3) or "").strip())
    return {"payloads":payload[:-1],"elapsed_s":elapsed,"returncode":process.returncode,
            "stderr_bytes":len(process.stderr.encode()),"child_reaped":True}

class GNU10ReplayPlayer:
    """Fresh seeded engine per move; chronological actions come from the arena's saved record."""
    def __init__(self, project, *, level=1, seed=20260924, command_timeout_s=2.0):
        self.project=Path(project).resolve();self.level=level;self.seed=seed
        if type(level) is not int or not 0<=level<=10: raise ValueError("GNU Go level must be 0..10")
        if type(seed) is not int or not 0<=seed<=2**31-1: raise ValueError("Invalid seed")
        if not math.isfinite(command_timeout_s) or command_timeout_s<=0: raise ValueError("Invalid timeout")
        self.command_timeout_s=float(command_timeout_s)
        self.identity=gnugo10_binary_identity(self.project);self.prepared=None
    def prepare_position(self, board, opening_actions, applied_actions):
        started=time.perf_counter()
        if board.size!=9 or board.terminal: raise ValueError("Reference requires a nonterminal 9x9 position")
        actions=list(opening_actions)+list(applied_actions)
        if len(actions)>2048: raise ValueError("Replay move limit exceeded")
        replay=initial_state(board.size,board.komi);commands=["boardsize 9","clear_board",f"komi {board.komi:g}"]
        for action in actions:
            color="b" if replay.to_play==BLACK else "w"
            commands.append(f"play {color} {gtp_action_to_vertex(action,9)}")
            replay=play(replay,action)
        if replay.to_dict()!=board.to_dict():
            raise ValueError("Chronological replay differs from complete supplied board/history; refusing incomplete superko")
        state_json=json.dumps(board.to_dict(),sort_keys=True,separators=(",",":"))
        call_seed=int(hashlib.sha256((str(self.seed)+state_json).encode()).hexdigest()[:8],16)%(2**31)
        self.prepared={"state":state_json,"commands":commands,"seed":call_seed,"started":started}
    def __call__(self, board, budget):
        if not math.isfinite(budget) or budget<=0: raise ValueError("Positive finite move budget required")
        record=self.prepared;self.prepared=None  # Preparation is consumed; never reuse a stale position.
        if record is None or record["state"]!=json.dumps(board.to_dict(),sort_keys=True,separators=(",",":")):
            raise ValueError("Call prepare_position with the complete saved chronology before each reference move")
        remaining=budget-(time.perf_counter()-record["started"])
        if remaining<=0: raise TimeoutError("Reference replay exhausted move budget; result UNKNOWN")
        color="b" if board.to_play==BLACK else "w"
        result=gnugo10_exchange(self.project,[*record["commands"],f"genmove {color}"],level=self.level,
            seed=record["seed"],timeout_s=min(self.command_timeout_s,remaining))
        action=gtp_vertex_to_action(result["payloads"][-1],board.size)
        child=play(board,action)
        if not verify_go_emission(board,action,child): raise RuntimeError("Reference move failed independent legality check")
        elapsed=time.perf_counter()-record["started"]
        return {"action":action,"status":"EXTERNAL_REFERENCE","strategic_status":"UNKNOWN",
            "deadline_missed":elapsed>budget,"elapsed_s":elapsed,"legality_verified":True,
            "child_reaped":True,"counters":{"gtp_commands":len(record["commands"])+2},
            "reference_seed":record["seed"],"reference_level":self.level}

def arena_gnugo_player(project,name="reference",*,level=1,seed=20260924,command_timeout_s=2.0):
    """Call after importing notebook 11 tagged definitions; never used for AHR training."""
    player_type=globals().get("FrozenArenaPlayer")
    if player_type is None: raise RuntimeError("Load notebook 11 ahr-export definitions before creating an arena player")
    implementation=GNU10ReplayPlayer(project,level=level,seed=seed,command_timeout_s=command_timeout_s)
    return player_type(name=name,decide=implementation,prepare_position=implementation.prepare_position,
        config={"engine":"GNU Go","level":level,"seed":seed,"work_budget":{},
                "rules":RULESET,"rule_flags":GNUGO10_RULE_FLAGS,"command_timeout_s":command_timeout_s,
                "timing":"whole call including chronological replay, process startup and cleanup; no hard realtime guarantee",
                "recovery":"fresh process and state-derived seed every call; history rebuilt from saved arena moves",
                "scoring":"project independent final area scorer; GNU Go final_score never used"},
        provenance={"kind":"external_engine","purpose":"evaluation_only",**implementation.identity,
                    "training_labels":False,"strength_rating":"UNASSIGNED"})


## Check protocol behaviour and rule boundaries

These assertions exercise observable protocol behaviour, including failed-command rollback, out-of-turn requests, coordinate round trips, captures, suicide, superko, undo and terminal scoring. Model quality is outside this notebook's scope.

In [ ]:
checks = {}
engine = AHRGTPEngine(gtp_fixture_player)
assert engine.handle_line("001 protocol_version\r\n") == "=1 2\n\n"
assert engine.handle_line("\t# comment\n") is None
assert engine.handle_line("pro\x00tocol_version\n") == "= 2\n\n"
assert engine.handle_line("8 known_command loadsgf") == "=8 false\n\n"
assert engine.handle_line("9 known_command genmove") == "=9 true\n\n"
assert engine.handle_line("10 NAME") == "?10 unknown command\n\n"
assert engine.handle_line("11") == "?11 unknown command\n\n"
assert set(engine.handle_line("list_commands")[2:].strip().splitlines()) == set(GTP_COMMANDS)
for size in range(2, 10):
    for action in range(size*size+1):
        assert gtp_vertex_to_action(gtp_action_to_vertex(action,size),size) == action
assert gtp_vertex_to_action("J9",9) == 8 and gtp_vertex_to_action("A1",9) == 72
checks["framing_comments_ids_coordinates"] = True

# Independent notebook namespaces produce distinct GoState classes. The adapter
# must rehydrate full history without leaking/replacing the engine's class.
foreign_namespace = {"__name__": "__gtp_class_identity_fixture__"}
foreign_book = nbformat.read(PROJECT / "notebooks/01_go_rules_and_independent_verifier.ipynb", as_version=4)
for foreign_cell in foreign_book.cells:
    if foreign_cell.cell_type == "code" and "ahr-export" in foreign_cell.metadata.get("tags", []):
        exec(compile(foreign_cell.source, "01:foreign-class-fixture", "exec"), foreign_namespace)
assert foreign_namespace["GoState"] is not GoState
bridge_seen = []
def foreign_player_fixture(state, model, time_budget_s, search_config):
    assert isinstance(state, foreign_namespace["GoState"])
    assert not isinstance(state, GoState)
    bridge_seen.append(state.to_dict())
    return {"action": state.pass_action, "status": "NAMESPACE_FIXTURE"}
foreign_namespace["choose_go_ahr_move"] = foreign_player_fixture
bridged_player = _gtp_bind_go_player(foreign_namespace, object())
bridge_state = play(initial_state(), 0)
assert bridged_player(bridge_state, time_budget_s=0.1)["action"] == bridge_state.pass_action
assert bridge_seen == [bridge_state.to_dict()]
checks["cross_namespace_model_state_bridge"] = True


before = engine.session_snapshot()
for command in ("boardsize 19", "boardsize -1", "komi nan", "komi inf", "play b I3",
                "play b A0", "play green A1", "time_left b 1.5 0", "time_settings 5 10 -1",
                "undo", "final_score", "play b A1 extra", "2147483648 name"):
    assert engine.handle_line(command).startswith("?"), command
    assert engine.session_snapshot() == before, command
checks["failed_commands_are_transactional"] = True

assert engine.handle_line("play b A1") == "=\n\n"
first = engine.state
assert engine.handle_line("play b B1") == "=\n\n"
assert engine.state.board[72:74] == (BLACK, BLACK)
assert engine.handle_line("undo") == "=\n\n"
assert engine.state == first
assert engine.handle_line("komi 3.5") == "=\n\n"
assert engine.handle_line("undo") == "=\n\n"
assert engine.state.komi == 3.5 and not any(engine.state.board)
assert engine.handle_line("play b pass") == "=\n\n"
assert engine.handle_line("play b pass") == "=\n\n" and not engine.state.terminal
assert engine.handle_line("play w pass") == "=\n\n" and engine.state.terminal
terminal = engine.state
assert engine.handle_line("final_score") == "= W+3.5\n\n"
assert engine.handle_line("genmove b") == "= pass\n\n" and engine.state == terminal
assert engine.handle_line("play b A1") == "? illegal move\n\n"
assert engine.handle_line("undo") == "=\n\n" and not engine.state.terminal
checks["out_of_turn_undo_komi_terminal"] = True

capture = AHRGTPEngine(gtp_fixture_player,board_size=3,komi=0)
for command in ("play w B2", "play b A2", "play b B1", "play b C2"):
    assert capture.handle_line(command) == "=\n\n"
capture_before = capture.state
assert capture.handle_line("play b B3") == "=\n\n"
assert capture.captures[BLACK] == 1 and capture.state.board[4] == EMPTY
assert capture.handle_line("play w B2") == "? illegal move\n\n"
assert capture.handle_line("undo") == "=\n\n"
assert capture.state == capture_before and capture.captures[BLACK] == 0
# A reachable ko fixture is unnecessary here: explicitly retain the candidate's
# board as historical evidence and ensure both checkers reject its repetition.
ko = AHRGTPEngine(gtp_fixture_player,board_size=3)
repeated = play(ko.state,0).board
ko.state = replace(ko.state, history=ko.state.history | {repeated})
ko_before = ko.session_snapshot()
assert ko.handle_line("play b A3") == "? illegal move\n\n"
assert ko.session_snapshot() == ko_before
checks["capture_suicide_superko_and_exact_undo"] = True

for invalid in (lambda s,time_budget_s: {"action": True},
                lambda s,time_budget_s: {"action": 999},
                lambda s,time_budget_s: {"action": None},
                lambda s,time_budget_s: 42):
    broken = AHRGTPEngine(invalid)
    before = broken.session_snapshot()
    assert broken.handle_line("55 genmove b").startswith("?55 ")
    assert broken.session_snapshot() == before

def failing_player(state,time_budget_s):
    raise RuntimeError("simulated callback failure")

broken = AHRGTPEngine(failing_player)
before = broken.session_snapshot()
assert broken.handle_line("genmove b").startswith("? ") and broken.session_snapshot() == before
checks["invalid_player_output_and_exceptions_are_transactional"] = True

budgets = []
def timed_fixture(state,time_budget_s):
    budgets.append(time_budget_s)
    return {"action": state.pass_action}
clocked = AHRGTPEngine(timed_fixture, default_time_budget_s=10)
assert clocked.handle_line("time_settings 60 20 5") == "=\n\n"
assert clocked.handle_line("time_left b 10 2") == "=\n\n"
assert clocked.handle_line("genmove b") == "= pass\n\n"
assert budgets[-1] == 4.5 and clocked.clocks[BLACK][1] == 1
assert clocked.handle_line("time_settings 0 0 0") == "=\n\n"
count = len(budgets)
assert clocked.handle_line("genmove w") == "= pass\n\n" and len(budgets) == count
assert clocked.handle_line("clear_board") == "=\n\n"
assert clocked.handle_line("time_settings 0 10 0") == "=\n\n"
assert clocked.time_budget(BLACK) == 10
assert clocked.handle_line("time_settings 0 10 5") == "=\n\n"
assert clocked.time_budget(BLACK) == 1.8
checks["absolute_canadian_unlimited_and_zero_clock"] = True

transcript_engine = AHRGTPEngine(gtp_fixture_player)
transcript_commands = ["1 boardsize 9", "2 clear_board", "3 komi 7.5", "4 play b D4",
                       "5 genmove w", "6 undo", "7 showboard", "8 quit"]
transcript = [(command, transcript_engine.handle_line(command)) for command in transcript_commands]
assert transcript_engine.closed and transcript_engine.handle_line("name") is None
assert all(response.startswith("=") and response.endswith("\n\n") for _,response in transcript)
checks["transcript_and_quit"] = True
for command,response in transcript:
    print("> " + command + "\n" + response.rstrip() + "\n")
print(f"PASS: {len(checks)} protocol and rules check groups")

## Verify a real standard-input/standard-output subprocess

This short process reads notebook exports in a fresh Python interpreter, answers a piped command sequence and exits on `quit`. Exact output equality checks for accidental startup logs, malformed framing and lost command IDs. The timeout prevents an orphaned protocol check.

In [ ]:
subprocess_input = "# fixture only\n21 protocol_version\n22 boardsize 9\n23 clear_board\n24 play black D4\n25 genmove white\n26 undo\n27 quit\n28 name\n"
expected_output = "=21 2\n\n=22\n\n=23\n\n=24\n\n=25 A9\n\n=26\n\n=27\n\n"
process = subprocess.run(make_gtp_subprocess_command(PROJECT), input=subprocess_input,
    text=True, capture_output=True, timeout=30, cwd=PROJECT, check=True)
assert process.stdout == expected_output, repr(process.stdout)
assert process.stderr == "", process.stderr
checks["fresh_process_stdio_exact_round_trip"] = True
print(process.stdout.rstrip())
print("PASS: subprocess exited cleanly; stdout contains only GTP frames.")

## Optionally check a selected Go checkpoint

Set `AHR10_CHECKPOINT_RUN` to an existing notebook 08 or compatible notebook 09 run ID and `AHR10_DEVICE` to the desired device before Run All. `AHR10_TIME_BUDGET_S` defaults to one second. The short local subprocess receives an existing stone and its superko history, generates a reply, rejects an occupied-point move, undoes the reply, checks the restored board and quits. With no selected run, the result is explicitly `NOT_RUN_NO_SELECTED_CHECKPOINT`. A bad checkpoint stops the run rather than substituting an untrained model.

In [ ]:
selected_go_run = os.environ.get("AHR10_CHECKPOINT_RUN", "").strip()
selected_selection = os.environ.get("AHR10_SELECTION", "latest")
selected_checkpoint_check = {"status": "NOT_RUN_NO_SELECTED_CHECKPOINT"}
if selected_go_run:
    selected_budget = float(os.environ.get("AHR10_TIME_BUDGET_S", "1.0"))
    selected_process = subprocess.run(make_go_gtp_subprocess_command(PROJECT, selected_go_run,
        device=os.environ.get("AHR10_DEVICE", "cpu"), selection=os.environ.get("AHR10_SELECTION", "latest"), default_time_budget_s=selected_budget),
        input="31 boardsize 9\n32 clear_board\n33 play b D4\n34 genmove w\n35 play b D4\n36 ahr-last-search\n37 undo\n38 showboard\n39 quit\n",
        text=True, capture_output=True, timeout=90, cwd=PROJECT, check=True)
    frames = selected_process.stdout.strip().split("\n\n")
    assert len(frames) == 9, selected_process.stdout
    for index, frame in enumerate(frames):
        assert frame.startswith(("?" if index == 4 else "=") + str(31+index)), selected_process.stdout
    assert frames[4] == "?35 illegal move"
    move = frames[3].split(maxsplit=1)[1]
    before_reply = play(initial_state(), gtp_vertex_to_action("D4",9))
    assert gtp_vertex_to_action(move,9) in legal_actions(before_reply)
    checked_child = play(before_reply,gtp_vertex_to_action(move,9))
    assert verify_go_emission(before_reply,gtp_vertex_to_action(move,9),checked_child)
    diagnostics = json.loads(frames[5].split(maxsplit=1)[1])
    assert diagnostics["legality_verified"] and diagnostics["player_identity"] == f"go_ahr:{selected_go_run}:{selected_selection}"
    assert diagnostics["checkpoint"]["run_id"] == selected_go_run and diagnostics["checkpoint"]["optimizer_step"] > 0
    assert len(diagnostics["checkpoint"]["checkpoint_sha256"]) == 64
    expected_undo = AHRGTPEngine(gtp_fixture_player)
    assert expected_undo.handle_line("play b D4") == "=\n\n"
    assert frames[7] == expected_undo.handle_line("38 showboard").rstrip()
    selected_checkpoint_check = {"status": "PASS", "run_id": selected_go_run, "move": move,
        "device": os.environ.get("AHR10_DEVICE", "cpu"), "time_budget_s": selected_budget,
        "legality_verified": True, "occupied_move_rejected": True, "undo_board_verified": True,
        "full_state_namespace_bridge": True, "search_diagnostics": diagnostics}
print(json.dumps(selected_checkpoint_check, indent=2))

## Save validation evidence

The report identifies the checked definitions and distinguishes protocol success from playing strength. No pretrained model is selected by Run All. The HTML preview is generated as part of delivery; graphical inspection is reported separately from execution and HTML structure checks.

In [ ]:
notebook = nbformat.read(NOTEBOOK_PATH, as_version=4)
nbformat.validate(notebook)
source_digest = hashlib.sha256("\n".join(c.source for c in notebook.cells if c.cell_type == "code").encode()).hexdigest()
report = {"schema": "ahr_gtp_validation_v1", "status": "PASS", "checks": checks,
    "notebook": str(NOTEBOOK_PATH.relative_to(PROJECT)), "source_sha256": source_digest,
    "upstream_export_sha256": SOURCE_HASHES, "protocol_version": 2,
    "implemented_commands": list(GTP_COMMANDS), "subprocess_exit_code": process.returncode,
    "fixture_only": not bool(selected_go_run), "trained_player_tested": bool(selected_go_run),
    "selected_checkpoint_check": selected_checkpoint_check, "optimizer_steps": 0,
    "rules": RULESET, "strength_claim": "NOT_EVALUATED",
    "terminal_genmove": "pass acknowledgment; no state transition",
    "deadline_semantics": "cooperative; controller adjudicates timeouts",
    "visual_inspection": "NOT_PERFORMED; open reports/previews/10_gtp_engine_interface.html in a graphical browser"}
report_path = PROJECT / "reports/10_gtp_validation.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
temporary = report_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
os.replace(temporary, report_path)
print(json.dumps({"status": report["status"], "check_groups": len(checks),
                  "report": str(report_path.relative_to(PROJECT)), "optimizer_steps": 0}, indent=2))

## Use a frozen player

In a notebook kernel, construct `AHRGTPEngine(your_frozen_player, supported_sizes=(9,))`, then send commands with `handle_line` or connect streams with `run`. The callable receives the complete immutable Go state, including superko history, and a budget in seconds; it returns an integer board action or the pass action. The interface independently checks legality and always labels strategic certainty `UNKNOWN`.

For an external GUI, obtain the argument list from `make_gtp_subprocess_command(PROJECT, factory_notebooks=(...), factory_name="your_exported_factory", factory_kwargs={...}, engine_kwargs={"supported_sizes": [9]})`. Pass this list directly to `subprocess.Popen`; GUI launch fields that require a shell string should use Python's `shlex.join` on the returned list. The factory and all reusable code remain in tagged notebook cells.

The protocol fixture proves communication and state handling. It does not establish AHR playing strength, a hard real-time guarantee, full GTP tournament-handicap support, or compatibility with every GUI. Match evaluation belongs in notebook 11.